# 第15章　実装ガイド：セグメンテーション ― MONAIで3Dを塗る

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 15.1　前処理を組み立てる ― MONAIのCompose

In [ ]:
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd,
    Orientationd, Spacingd, ScaleIntensityRanged, CropForegroundd, RandCropByPosNegLabeld)

train_tf = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),           # 向きを統一
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5),
             mode=("bilinear", "nearest")),                         # 画像は補間/ラベルは最近傍
    ScaleIntensityRanged(keys="image", a_min=-160, a_max=240,
                         b_min=0.0, b_max=1.0, clip=True),          # 腹部ウィンドウ（HU値）
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(keys=["image", "label"], label_key="label",
        spatial_size=(96, 96, 96), pos=2, neg=1, num_samples=4),    # 病変中心を多くサンプリング
    # 「陽性」は label_key の非零ボクセル。3クラスのラベルをそのまま渡すと肝臓全体が陽性になるので、
    # 腫瘍を重点的に切り出したいなら、腫瘍だけを1にしたマスクを別のキーで用意して label_key に渡す
])

val_tf = Compose([                                                  # 検証用：乱数を使う変換は入れない
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5),
             mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys="image", a_min=-160, a_max=240,
                         b_min=0.0, b_max=1.0, clip=True),
])                                    # クロップしないのは、推論をスライディングウィンドウで行うため（15.3）

## データを供給する ― MONAIのDataLoaderを使う

In [ ]:
from monai.data import CacheDataset, DataLoader   # ← torch ではなく monai の DataLoader

# 症例を「辞書のリスト」で並べる。キーは transform の keys と一致させる
train_files = [{"image": f"imagesTr/case_{i:03d}.nii.gz",
                "label": f"labelsTr/case_{i:03d}.nii.gz"} for i in range(80)]
val_files   = [{"image": f"imagesTr/case_{i:03d}.nii.gz",
                "label": f"labelsTr/case_{i:03d}.nii.gz"} for i in range(80, 100)]

train_ds = CacheDataset(train_files, transform=train_tf, cache_rate=1.0)  # 前処理結果を再利用
val_ds   = CacheDataset(val_files,   transform=val_tf,   cache_rate=1.0)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=1, num_workers=2)

## 15.2　モデルと損失

In [ ]:
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet(spatial_dims=3, in_channels=1, out_channels=2,           # 背景+病変の2クラス（章頭の約束。肝・腫瘍を分けるなら3）
             channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)).to(device)
criterion = DiceCELoss(to_onehot_y=True, softmax=True)               # Dice + 交差エントロピー
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

## 15.3　学習と、スライディングウィンドウ推論

In [ ]:
for epoch in range(50):
    model.train()
    for batch in train_loader:
        img, lbl = batch["image"].to(device), batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(img), lbl)      # DiceCELoss が Dice と CE をまとめて計算
        loss.backward()
        optimizer.step()

In [ ]:
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
from monai.data import decollate_batch

# 規約：正解が空（病変なし）の症例は Dice を NaN にして平均から除く（ignore_empty=True。
# MONAI の既定値でもあるが、規約なので明示する）。除いた症例は、別の指標で必ず報告する
dice = DiceMetric(include_background=False, ignore_empty=True)
post_pred  = AsDiscrete(argmax=True, to_onehot=2)   # 予測: argmax→one-hot [C,...]
post_label = AsDiscrete(to_onehot=2)                # 正解: one-hot [C,...]
n_neg, n_neg_fp = 0, 0                              # 正解が空の症例数と、そこへ前景を出した症例数
model.eval()
with torch.no_grad():
    for batch in val_loader:
        img, lbl = batch["image"].to(device), batch["label"].to(device)
        out = sliding_window_inference(img, roi_size=(96, 96, 96),
                                       sw_batch_size=2, predictor=model, overlap=0.5)
        # DiceMetric は one-hot [B,C,...] を前提。各サンプルを one-hot 化して渡す
        preds  = [post_pred(o)  for o in decollate_batch(out)]
        labels = [post_label(l) for l in decollate_batch(lbl)]
        for pr, lb in zip(preds, labels):           # 陰性例の偽陽性は Dice の平均に現れないので別に数える
            if lb[1].sum() == 0:
                n_neg += 1; n_neg_fp += int(pr[1].sum() > 0)
        dice(y_pred=preds, y=labels)
print(f"GT陽性例の前景Dice: {dice.aggregate().item():.3f}  "
      f"/ 正解が空の症例 {n_neg}例（うち偽陽性あり {n_neg_fp}例）")   # 病変別・大きさ別にも見る

## 15.4　手早く高精度を狙うなら ― nnU-Net

```bash
nnUNetv2_plan_and_preprocess -d 3 --verify_dataset_integrity   # 検証+自動設計（Dataset003_Liver）
nnUNetv2_train 3 3d_fullres 0                                  # まずは fold 0 だけ（既定1000エポック）
# for f in 0 1 2 3 4; do nnUNetv2_train 3 3d_fullres $f; done  # 5分割すべて＝アンサンブル用
nnUNetv2_predict -i imagesTs/ -o preds/ -d 3 -c 3d_fullres -f 0   # 推論（学習したfoldを指定）
```

## nnU-Netの流儀 ― 環境変数・フォルダ規約・dataset.json

```bash
export nnUNet_raw="/data/nnUNet_raw"
export nnUNet_preprocessed="/data/nnUNet_preprocessed"
export nnUNet_results="/data/nnUNet_results"
```

```text
nnUNet_raw/Dataset003_Liver/
├── imagesTr/    liver_0_0000.nii.gz, liver_1_0000.nii.gz, ...   （学習画像）
├── labelsTr/    liver_0.nii.gz,      liver_1.nii.gz, ...        （学習ラベル）
├── imagesTs/    liver_131_0000.nii.gz, ...   （任意・最終テスト用の画像）
└── dataset.json
```

```json
{
  "channel_names": {"0": "CT"},
  "labels": {"background": 0, "liver": 1, "tumor": 2},
  "numTraining": 131,
  "file_ending": ".nii.gz"
}
```